In [11]:
sentences = [
    "Movie is good",
    "Movie is bad",
    "Movie is very bad"
]
vocab = ["<UNK>", "<END>"]

# Build vocabulary
for sentence in sentences:
    words = sentence.lower().split()
    for word in words:
        if word not in vocab:
            vocab.append(word)

# Create word-to-index mapping
word_to_index = {}
for index, word in enumerate(vocab):
    word_to_index[word] = index
vocab_size = len(vocab)

print("One-Hot Encoding:\n")

for sentence in sentences:
    words = sentence.lower().split()
    print(f"--- Processing: '{sentence}' ---")
    
    for word in words:
        # Use word_to_index dictionary directly for safety and speed
        word_index = word_to_index[word]
        
        # Create a blank vector of zeros with length equal to vocab_size
        one_hot_vector = [0] * vocab_size
        
        # Set the correct position to 1
        one_hot_vector[word_index] = 1
        
        # Print the word and its vector
        print(f"{word:<6} -> {one_hot_vector}")
    print()

One-Hot Encoding:

--- Processing: 'Movie is good' ---
movie  -> [0, 0, 1, 0, 0, 0, 0]
is     -> [0, 0, 0, 1, 0, 0, 0]
good   -> [0, 0, 0, 0, 1, 0, 0]

--- Processing: 'Movie is bad' ---
movie  -> [0, 0, 1, 0, 0, 0, 0]
is     -> [0, 0, 0, 1, 0, 0, 0]
bad    -> [0, 0, 0, 0, 0, 1, 0]

--- Processing: 'Movie is very bad' ---
movie  -> [0, 0, 1, 0, 0, 0, 0]
is     -> [0, 0, 0, 1, 0, 0, 0]
very   -> [0, 0, 0, 0, 0, 0, 1]
bad    -> [0, 0, 0, 0, 0, 1, 0]



# Charecter level Encoding

In [11]:
import torch
import torch.nn as nn

In [12]:
text = """
Artificial intelligence is changing the world.
Machine learning helps computers learn from data.
Deep learning uses neural networks to solve complex problems.
Recurrent neural networks process sequential information.
They remember information from previous steps using hidden states.
Natural language processing helps machines understand human language.
Text generation predicts the next character based on previous characters.
Learning requires data, patience, and repeated experiments.
Technology can help people solve important problems.
Researchers continuously improve intelligent systems.
""" * 20

In [13]:
text

'\nArtificial intelligence is changing the world.\nMachine learning helps computers learn from data.\nDeep learning uses neural networks to solve complex problems.\nRecurrent neural networks process sequential information.\nThey remember information from previous steps using hidden states.\nNatural language processing helps machines understand human language.\nText generation predicts the next character based on previous characters.\nLearning requires data, patience, and repeated experiments.\nTechnology can help people solve important problems.\nResearchers continuously improve intelligent systems.\n\nArtificial intelligence is changing the world.\nMachine learning helps computers learn from data.\nDeep learning uses neural networks to solve complex problems.\nRecurrent neural networks process sequential information.\nThey remember information from previous steps using hidden states.\nNatural language processing helps machines understand human language.\nText generation predicts the n

In [15]:
chars=sorted(set(text))
char_to_id={ch:i for i,ch in enumerate(chars)}
id_to_char={i:ch for i,ch in enumerate(chars)}

vocab_len=len(chars)
print(vocab_len)
print(id_to_char)

35
{0: '\n', 1: ' ', 2: ',', 3: '.', 4: 'A', 5: 'D', 6: 'L', 7: 'M', 8: 'N', 9: 'R', 10: 'T', 11: 'a', 12: 'b', 13: 'c', 14: 'd', 15: 'e', 16: 'f', 17: 'g', 18: 'h', 19: 'i', 20: 'k', 21: 'l', 22: 'm', 23: 'n', 24: 'o', 25: 'p', 26: 'q', 27: 'r', 28: 's', 29: 't', 30: 'u', 31: 'v', 32: 'w', 33: 'x', 34: 'y'}


In [16]:
encoded=torch.tensor([char_to_id[char] for char in text])
print(encoded[:20])

tensor([ 0,  4, 27, 29, 19, 16, 19, 13, 19, 11, 21,  1, 19, 23, 29, 15, 21, 21,
        19, 17])


In [18]:
seq_length=40

X=[]
Y=[]

for i in range(len(encoded)-seq_length):
    X.append(encoded[i:i+seq_length])
    Y.append(encoded[i+seq_length])
X = torch.stack(X)
Y = torch.stack(Y)
print(X[0])
print(Y[0])

tensor([ 0,  4, 27, 29, 19, 16, 19, 13, 19, 11, 21,  1, 19, 23, 29, 15, 21, 21,
        19, 17, 15, 23, 13, 15,  1, 19, 28,  1, 13, 18, 11, 23, 17, 19, 23, 17,
         1, 29, 18, 15])
tensor(1)


In [20]:
X = nn.functional.one_hot(X, num_classes=vocab_len)
Y = nn.functional.one_hot(Y, num_classes=vocab_len)

In [21]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Define the RNN Model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_dim, n_layers=1):
        super(CharRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        # RNN layer
        self.rnn = nn.RNN(vocab_size, hidden_dim, n_layers, batch_first=True)
        # Fully connected output layer
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.n_layers, x.size(0), self.hidden_dim).to(x.device)
        
        # Forward pass
        out, hn = self.rnn(x, h0)
        # Take the output of the last time step
        out = self.fc(out[:, -1, :])
        return out

# Convert inputs/targets to correct PyTorch float types for training
# X has shape (num_sequences, seq_length, vocab_size) and should be float
# Y has shape (num_sequences, vocab_size) but CrossEntropyLoss expects targets as class indices of shape (num_sequences,)
# So we will revert Y back to integer index representation for PyTorch's CrossEntropyLoss
Y_indices = torch.argmax(Y, dim=1)

# Create a DataLoader for batching
dataset = TensorDataset(X.float(), Y_indices)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [23]:
# Initialize model, loss function, and optimizer
hidden_dim = 128
model = CharRNN(vocab_len, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train the model
epochs = 15
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in dataloader:
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_x.size(0)
    
    epoch_loss /= len(dataloader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/15], Loss: 1.8824
Epoch [2/15], Loss: 0.2437
Epoch [3/15], Loss: 0.0208
Epoch [4/15], Loss: 0.0076
Epoch [5/15], Loss: 0.0024
Epoch [6/15], Loss: 0.0016
Epoch [7/15], Loss: 0.0011
Epoch [8/15], Loss: 0.0009
Epoch [9/15], Loss: 0.0007
Epoch [10/15], Loss: 0.0006
Epoch [11/15], Loss: 0.0005
Epoch [12/15], Loss: 0.0004
Epoch [13/15], Loss: 0.0003
Epoch [14/15], Loss: 0.0003
Epoch [15/15], Loss: 0.0002


In [25]:
def generate_text(model, seed_text, gen_length=100):
    model.eval()
    generated = seed_text
    
    with torch.no_grad():
        for _ in range(gen_length):
            # Format the current seed text to match the input sequence length
            # If seed text is shorter than seq_length, pad it. If longer, slice the last seq_length characters.
            if len(seed_text) < seq_length:
                padded_seed = seed_text.rjust(seq_length, ' ')
            else:
                padded_seed = seed_text[-seq_length:]
            
            # Convert to numeric IDs
            input_ids = torch.tensor([char_to_id[ch] for ch in padded_seed])
            
            # One-hot encode the inputs
            input_one_hot = nn.functional.one_hot(input_ids, num_classes=vocab_len).float()
            
            # Add batch dimension (1, seq_length, vocab_size)
            input_tensor = input_one_hot.unsqueeze(0)
            
            # Get model prediction
            output = model(input_tensor)
            
            # Find the index with the highest probability
            predicted_id = torch.argmax(output, dim=1).item()
            predicted_char = id_to_char[predicted_id]
            
            # Append to generated text and update the seed
            generated += predicted_char
            seed_text += predicted_char
            
    return generated

# Test the model with a seed phrase
seed = "Artificial intelligence is "
generated_output = generate_text(model, seed, gen_length=150)
print("Generated Text:\n", generated_output)

Generated Text:
 Artificial intelligence is changing the world.
Machine learning helps computers learn from data.
Deep learning uses neural networks to solve complex problems.
Recurrent neural n


# Word level encoding

In [27]:
import torch
import torch.nn as nn

In [28]:
text = """
Artificial intelligence is changing the world.
Machine learning helps computers learn from data.
Deep learning uses neural networks to solve complex problems.
Recurrent neural networks process sequential information.
They remember information from previous steps using hidden states.
Natural language processing helps machines understand human language.
Text generation predicts the next character based on previous characters.
Learning requires data, patience, and repeated experiments.
Technology can help people solve important problems.
Researchers continuously improve intelligent systems.
""" * 20

# What is enumerate() doing?
enumerate(unique_words) pairs each element in your list with its index position, creating a series of (index, item) pairs:

(0, 'and')
(1, 'learning')
(2, 'patience')
(3, 'requires')
2. The for i, word in ... Part
The expression for i, word in enumerate(unique_words) loops through those pairs and splits (unpacks) them on every turn:

In iteration 1: i = 0, word = 'and'
In iteration 2: i = 1, word = 'learning'
In iteration 3: i = 2, word = 'patience'
In iteration 4: i = 3, word = 'requires'
3. The Mapping: {word: i ...}
A Python dictionary is structured as {key: value}. In {word: i for i, word in ...}:

Key is word (the string value)
Value is i (the numerical index position)
This maps each unique string word directly to its assigned integer ID, yielding:

In [34]:
import re

cleaned_text=re.sub(r'[.,\n]'," ",text.lower())
words=[word for word in cleaned_text.split(" ") if word ]

unique_words=sorted(list(set(words)))
word_vocab_size=len(unique_words)
word_to_id={word:i for i,word in enumerate(unique_words)}
id_to_words={i:word for i,word in enumerate(unique_words)}

print(f"Total Words in text {len(words)}")
print(f"Total unique words in vocab:{word_vocab_size}")
print(f'unique words,{unique_words}\n')
print("Sample vocabulary mappings:", list(word_to_id.items())[:61])

Total Words in text 1480
Total unique words in vocab:61
unique words,['and', 'artificial', 'based', 'can', 'changing', 'character', 'characters', 'complex', 'computers', 'continuously', 'data', 'deep', 'experiments', 'from', 'generation', 'help', 'helps', 'hidden', 'human', 'important', 'improve', 'information', 'intelligence', 'intelligent', 'is', 'language', 'learn', 'learning', 'machine', 'machines', 'natural', 'networks', 'neural', 'next', 'on', 'patience', 'people', 'predicts', 'previous', 'problems', 'process', 'processing', 'recurrent', 'remember', 'repeated', 'requires', 'researchers', 'sequential', 'solve', 'states', 'steps', 'systems', 'technology', 'text', 'the', 'they', 'to', 'understand', 'uses', 'using', 'world']

Sample vocabulary mappings: [('and', 0), ('artificial', 1), ('based', 2), ('can', 3), ('changing', 4), ('character', 5), ('characters', 6), ('complex', 7), ('computers', 8), ('continuously', 9), ('data', 10), ('deep', 11), ('experiments', 12), ('from', 13), ('ge

Correct Sequence: ["artificial", "intelligence", "is", "changing", "the", "world"]
If we used only unique_words to make sequences, we would completely destroy the natural context, sentence structures, and repeating patterns of human language because words would only appear once in alphabetical order.

In [35]:
Encoded_words=torch.tensor([word_to_id[w]for w in words])
word_seq_length=5
X_word=[]
Y_word=[]

for i in range(0,len(Encoded_words)-word_seq_length):
    X_word.append(Encoded_words[i:i+word_seq_length])
    Y_word.append(Encoded_words[i+word_seq_length])
X_word=torch.stack(X_word)
Y_word=torch.stack(Y_word)

print(f"Input shape:{X_word.shape}")
print(f"Target shape:{Y_word.shape}")

#printing the first sequence
first_seq=[id_to_words[idx.item()]for idx in X_word[0]]
first_target=id_to_words[Y_word[0].item()]
print(f"First sequence: {first_seq} -> Next word: {first_target}")

Input shape:torch.Size([1475, 5])
Target shape:torch.Size([1475])
First sequence: ['artificial', 'intelligence', 'is', 'changing', 'the'] -> Next word: world


In [36]:
X_word_one_hot = nn.functional.one_hot(X_word,num_classes=word_vocab_size).float()
Y_word_one_hot = nn.functional.one_hot(Y_word, num_classes=word_vocab_size).float()

word_dataset_onehot = TensorDataset(X_word_one_hot, Y_word_one_hot)
word_dataloader_onehot = DataLoader(word_dataset_onehot, batch_size=32, shuffle=True)

print(f"X_word_one_hot shape: {X_word_one_hot.shape}")
print(f"Y_word_one_hot shape: {Y_word_one_hot.shape}")

X_word_one_hot shape: torch.Size([1475, 5, 61])
Y_word_one_hot shape: torch.Size([1475, 61])


Think of the TensorDataset as a binder holding paired question-and-answer flashcards together, and the DataLoader as a study assistant who grabs a random stack of 32 flashcards at a time to hand to you during your study session.

In [37]:
import torch.optim as optim

# 1. Define the Word-Level RNN Model
class WordRNN(nn.Module):
    def __init__(self, vocab_size, hidden_dim, n_layers=1):
        super(WordRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        # RNN layer takes (batch, seq, vocab_size)
        self.rnn = nn.RNN(vocab_size, hidden_dim, n_layers, batch_first=True)
        # Fully connected output layer to predict logits over word vocabulary
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        h0 = torch.zeros(self.n_layers, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

# Initialize model, loss function, and optimizer
word_hidden_dim = 128
word_model = WordRNN(word_vocab_size, word_hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(word_model.parameters(), lr=0.01)

print("Model structure:")
print(word_model)    

Model structure:
WordRNN(
  (rnn): RNN(61, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=61, bias=True)
)


In [40]:
# 2. Train the Word-Level RNN
epochs = 15
for epoch in range(epochs):
    epoch_loss = 0.0
    # Use the correct dataloader defined in cell 8ec0236b
    for batch_x, batch_y in word_dataloader_onehot:
        # Convert one-hot target to class index for CrossEntropyLoss
        batch_y_indices = torch.argmax(batch_y, dim=1)
        
        # Forward pass
        outputs = word_model(batch_x)
        loss = criterion(outputs, batch_y_indices)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    epoch_loss /= len(word_dataloader_onehot.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/15], Loss: 0.0036
Epoch [2/15], Loss: 0.0014
Epoch [3/15], Loss: 0.0010
Epoch [4/15], Loss: 0.0007
Epoch [5/15], Loss: 0.0006
Epoch [6/15], Loss: 0.0005
Epoch [7/15], Loss: 0.0004
Epoch [8/15], Loss: 0.0004
Epoch [9/15], Loss: 0.0003
Epoch [10/15], Loss: 0.0003
Epoch [11/15], Loss: 0.0002
Epoch [12/15], Loss: 0.0002
Epoch [13/15], Loss: 0.0002
Epoch [14/15], Loss: 0.0002
Epoch [15/15], Loss: 0.0002


In [46]:
# 3. Define Word-Level Text Generator
def generate_word_text(model, seed_words, gen_length=20):
    model.eval()
    # Convert seed_words (a string) into a list of cleaned words
    seed_list = [w for w in re.sub(r'[.,\n]', ' ', seed_words.lower()).split(' ') if w]
    generated = list(seed_list)

    with torch.no_grad():
        for _ in range(gen_length):
            # Format current seed list to match input sequence length
            if len(seed_list) < word_seq_length:
                # If too short, pad with a common word or first word in vocabulary
                padded_seed = [id_to_words[0]] * (word_seq_length - len(seed_list)) + seed_list
            else:
                padded_seed = seed_list[-word_seq_length:]

            # Convert list of words to numeric indices
            input_ids = torch.tensor([word_to_id[w] for w in padded_seed])

            # One-hot encode inputs
            input_one_hot = nn.functional.one_hot(input_ids, num_classes=word_vocab_size).float()

            # Add batch dimension
            input_tensor = input_one_hot.unsqueeze(0)

            # Predict next word logits
            output = model(input_tensor)
            predicted_id = torch.argmax(output, dim=1).item()
            predicted_word = id_to_words[predicted_id]

            # Append prediction to seed and results
            generated.append(predicted_word)
            seed_list.append(predicted_word)

    return " ".join(generated)

# Test generating word predictions
seed_phrase = "artificial intelligence is"
generated_output = generate_word_text(word_model, seed_phrase, gen_length=15)
print("Generated Text:")
print(generated_output)

Generated Text:
artificial intelligence is help help world machine learning researchers researchers learn from repeated artificial artificial uses neural people


# Tiny Shakespeare dataset, clean and inspect it, and get it ready to plug straight into your RNN model:

In [48]:
import re
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Download the Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
print("Downloading dataset...")
text = urllib.request.urlopen(url).read().decode("utf-8")

# Let's check how big it is (it's around 1.1 million characters)
print(f"Original text length (characters): {len(text)}")

# Optional: Slice it down if you want faster training on your local machine
# (e.g., taking the first 50,000 characters so it runs quickly)
text = text[:50000]

# 2. Tokenize and Build Vocabulary
cleaned_text = re.sub(r"[.,\n\r;!?]", " ", text.lower())
words = [word for word in cleaned_text.split(" ") if word]

unique_words = sorted(list(set(words)))
word_vocab_size = len(unique_words)
word_to_id = {word: i for i, word in enumerate(unique_words)}
id_to_words = {i: word for i, word in enumerate(unique_words)}

print(f"Total words in text: {len(words)}")
print(f"Total unique words in vocab: {word_vocab_size}")

# 3. Create Sequences
Encoded_words = torch.tensor([word_to_id[w] for w in words])
word_seq_length = 5
X_word = []
Y_word = []

for i in range(0, len(Encoded_words) - word_seq_length):
    X_word.append(Encoded_words[i : i + word_seq_length])
    Y_word.append(Encoded_words[i + word_seq_length])


X_word = torch.stack(X_word)
Y_word = torch.stack(Y_word)

print(f"Input shape: {X_word.shape}")
print(f"Target shape: {Y_word.shape}")

# 4. One-Hot Encoding and Dataloader
X_word_one_hot = nn.functional.one_hot(
    X_word, num_classes=word_vocab_size
).float()
Y_word_one_hot = nn.functional.one_hot(
    Y_word, num_classes=word_vocab_size
).float()

word_dataset_onehot = TensorDataset(X_word_one_hot, Y_word_one_hot)
word_dataloader_onehot = DataLoader(
    word_dataset_onehot, batch_size=64, shuffle=True
)


# 5. Define the RNN Model
class WordRNN(nn.Module):

    def __init__(self, vocab_size, hidden_dim, n_layers=1):
        super(WordRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.rnn = nn.RNN(vocab_size, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.n_layers, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

# Initialize model, loss, and optimizer
word_hidden_dim = 256  # Slightly larger hidden dim for a bigger vocabulary
word_model = WordRNN(word_vocab_size, word_hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(word_model.parameters(), lr=0.005)

print("\nModel structure initialized successfully!")

Original text length (characters): 1115394
Total words in text: 8979
Total unique words in vocab: 2217
Input shape: torch.Size([8974, 5])
Target shape: torch.Size([8974])

Model structure initialized successfully!


In [49]:
# 2. Train the Word-Level RNN
epochs = 15
for epoch in range(epochs):
    epoch_loss = 0.0
    # Use the correct dataloader defined in cell 8ec0236b
    for batch_x, batch_y in word_dataloader_onehot:
        # Convert one-hot target to class index for CrossEntropyLoss
        batch_y_indices = torch.argmax(batch_y, dim=1)
        
        # Forward pass
        outputs = word_model(batch_x)
        loss = criterion(outputs, batch_y_indices)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    epoch_loss /= len(word_dataloader_onehot.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/15], Loss: 6.7768
Epoch [2/15], Loss: 5.9757
Epoch [3/15], Loss: 5.1855
Epoch [4/15], Loss: 4.1133
Epoch [5/15], Loss: 2.7872
Epoch [6/15], Loss: 1.7175
Epoch [7/15], Loss: 1.0979
Epoch [8/15], Loss: 0.7479
Epoch [9/15], Loss: 0.5594
Epoch [10/15], Loss: 0.3959
Epoch [11/15], Loss: 0.3886
Epoch [12/15], Loss: 0.3490
Epoch [13/15], Loss: 0.3192
Epoch [14/15], Loss: 0.4068
Epoch [15/15], Loss: 0.5762


In [52]:
import re
import torch
import torch.nn as nn


def generate_word_text(
    model, seed_words, gen_length=20, word_seq_length=5, word_vocab_size=61
):
    model.eval()
    # Clean and tokenize the input seed phrase
    seed_list = [
        w for w in re.sub(r"[.,\n\r;!?]", " ", seed_words.lower()).split(" ") if w
    ]
    generated = list(seed_list)

    with torch.no_grad():
        for _ in range(gen_length):
            # Format current seed list to match input sequence length
            if len(seed_list) < word_seq_length:
                # If too short, pad with the first word in vocabulary
                padded_seed = [id_to_words[0]] * (
                    word_seq_length - len(seed_list)
                ) + seed_list
            else:
                padded_seed = seed_list[-word_seq_length:]

            # Convert list of words to numeric indices (safely handling unknown words if needed)
            input_ids = torch.tensor([word_to_id.get(w, 0) for w in padded_seed])

            # One-hot encode inputs
            input_one_hot = nn.functional.one_hot(
                input_ids, num_classes=word_vocab_size
            ).float()

            # Add batch dimension: shape (1, seq_length, vocab_size)
            input_tensor = input_one_hot.unsqueeze(0)

            # Predict next word logits
            output = model(input_tensor)
            predicted_id = torch.argmax(output, dim=1).item()
            predicted_word = id_to_words[predicted_id]

            # Append prediction to seed and results
            generated.append(predicted_word)
            seed_list.append(predicted_word)

    return " ".join(generated)


# --- Test the Generator ---
# Pick a seed phrase that exists (or is likely to exist) in your Shakespeare vocabulary
seed_phrase = "We are accounted poor"

# Make sure to pass your actual word_vocab_size variable if it changed
generated_output = generate_word_text(
    word_model,
    seed_phrase,
    gen_length=25,
    word_seq_length=word_seq_length,
    word_vocab_size=word_vocab_size,
)

print("Generated Shakespeare Text:")
print(generated_output)

Generated Shakespeare Text:
we are accounted poor citizens the patricians good what authority surfeits on would relieve us: if they would yield us but the superfluity while it were wholesome we might
